# Zoomy in your browser

This notebook runs entirely in your browser via Pyodide — no install, no server,
no backend. `zoomy_core` and `zoomy_plotting` are bundled into the page.

It walks the whole stack once, on a **Shallow Moment** model that resolves a
vertical velocity profile:

**`Model`** (symbolic PDE) → **`SystemModel`** (frozen operators) →
**`NumericalSystemModel`** (+ scheme) → **C++ codegen** *and* **a NumPy solve** →
the reconstructed **velocity profile**.

Everything is symbolic until the very last step.

## 1. Model — a symbolic PDE

We use `SME(level=2)` — the Shallow Moment Equations carrying **two velocity
moments beyond the depth average**, so the solution has a real *vertical
velocity profile*, not one depth-averaged value. Physics enters as closures plus
two parameters: a bulk viscosity `nu` and a bed slip `lambda_s`. Without them the
profile is flat — the slip sets up the shear at the bed and the viscosity shapes
it through the depth.

In [ ]:
import numpy as np

from zoomy_core.model.models import SME, Newtonian, NavierSlip, StressFree
import zoomy_core.model.boundary_conditions as BC
import zoomy_core.model.initial_conditions as IC

model = SME(
    level=2,                                   # two moments beyond the depth average
    parameters={"nu": 0.1, "lambda_s": 0.5},   # bulk viscosity + bed slip
    closures=[Newtonian(), NavierSlip(), StressFree()],
    boundary_conditions=BC.BoundaryConditions([
        BC.Wall(tag="left"), BC.Wall(tag="right"),   # closed box
    ]),
    initial_conditions=IC.RP(                  # dam break, h = 2 | 1
        high=lambda n: np.array([0.0, 2.0] + [0.0] * (n - 2)),
        low=lambda n: np.array([0.0, 1.0] + [0.0] * (n - 2)),
        jump_position_x=5.0,
    ),
)
print(type(model).__name__, "with", [type(c).__name__ for c in model.closures])

## 2. SystemModel — freeze the operators

`SystemModel.from_model` walks the model once and freezes the operator matrices
into the canonical balance law

$$M\,\partial_t Q + \nabla\cdot(F + P) + \sum_d B_d\,\partial_d Q - S = 0.$$

`Model` and `SystemModel` are **siblings**, not parent and child — the second is
a frozen snapshot of the first.

In [ ]:
from zoomy_core.systemmodel import SystemModel

sm = SystemModel.from_model(model)

# Display, do NOT print: describe() has a _repr_markdown_ that renders the
# operator matrices as real LaTeX. print() would dump the raw markup.
sm.describe()

In [ ]:
print("state     :", [str(s) for s in sm.state])
print("aux state :", [str(s) for s in sm.aux_state])
print("flux      :", sm.flux.shape, " NCP:", sm.nonconservative_matrix.shape)

The aux state was not declared by anyone — the `d…dx` entries are the spatial
derivatives the operators need (one per state variable), discovered and
registered automatically. The solver fills them each step with one least-squares
gradient pass, so no finite-difference stencil is ever written by hand.

## 3. NumericalSystemModel — add the scheme

The scheme choices (Riemann solver, reconstruction, regularisation) live here,
at the symbolic level. Every backend reads them from this one place, which is
why no backend carries its own numerical constants.

In [ ]:
from zoomy_core.numerics import NumericalSystemModel, ReconstructionSpec

sm.aux_initial_conditions = IC.Constant(constants=lambda n: np.zeros(n))
nsm = NumericalSystemModel.from_system_model(
    sm, reconstruction=ReconstructionSpec(order=1))

riemann = nsm.riemann
print("Riemann       :", getattr(riemann, "__name__", None) or type(riemann).__name__)
print("reconstruction:", nsm.reconstruction)

## 4. Generate C++

The same object that is about to be solved in NumPy can be printed as C++. This
is the code that the compiled backends build against — generated here, in your
browser.

In [ ]:
from zoomy_core.transformation.to_c import CppModel

cpp = CppModel(nsm).create_code()
print(f"generated {len(cpp):,} characters of C++\n")
print("\n".join(cpp.splitlines()[:25]))

## 5. Solve, in the browser

50 cells on `x in [0, 10]`, adaptive timestep, to `t = 0.5`.

In [ ]:
from zoomy_core.mesh import BaseMesh
from zoomy_core.fvm.solver_numpy import HyperbolicSolver
from zoomy_core.fvm import timestepping

mesh = BaseMesh.create_1d(domain=(0.0, 10.0), n_inner_cells=50)
solver = HyperbolicSolver(time_end=0.5,
                          compute_dt=timestepping.adaptive(CFL=0.9))
Q, Qaux = solver.solve(mesh, nsm, write_output=False)

n = mesh.n_inner_cells
Q = np.asarray(Q[:, :n], dtype=float)
b, h = Q[0], Q[1]              # moments q_0, q_1, q_2 are Q[2], Q[3], Q[4]
print(f"h range : {h.min():.6f} .. {h.max():.6f}")

### A real check

The box is closed, so mass must be conserved exactly: 25 cells at `h = 2` and 25
at `h = 1` gives 75.

In [ ]:
error = abs(h.sum() - 75.0)
print(f"mass = {h.sum():.10f}   error = {error:.3e}")
assert error < 1e-9, "wall boundaries are leaking mass"

## 6. Plot — free surface and velocity profile

In [ ]:
import matplotlib.pyplot as plt
import zoomy_plotting as zp
from zoomy_core.model.derivation.basisfunctions import Legendre_shifted

x = mesh.cell_centers_computed()[0][:n]
order = np.argsort(x)

# Vertical velocity profile from the moments — the same reconstruction the
# coupling contract (and the GUI) use: u(zeta) = sum_i (q_i/h) phi_i(zeta),
# with zeta = 0 the bed and zeta = 1 the free surface.
basis = Legendre_shifted(2)
zeta = np.linspace(0.0, 1.0, 60)
stations = [4.0, 5.5, 7.0]
cells = [int(np.argmin(np.abs(x - xs))) for xs in stations]

with zp.apply_style():
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(9.5, 3.6))
    # left: the free surface, with the profile stations marked
    zp.line_plot(ax0, [
        {"x": x[order], "y": (b + h)[order], "label": "surface b + h", "role": "water"},
    ], xlabel="x", ylabel="elevation", title="free surface at t = 0.5")
    for k in cells:
        ax0.axvline(x[k], color="0.6", lw=1.0, ls="--")
    ax0.legend()
    # right: the vertical velocity profile at each station
    for k in cells:
        alpha = [Q[2 + j][k] / h[k] for j in range(3)]     # (q_0, q_1, q_2) / h
        ax1.plot(basis.reconstruct_velocity_profile(alpha, N=zeta.size), zeta,
                 lw=2, label=f"x = {x[k]:.1f}")
    ax1.set_xlabel("u(zeta)")
    ax1.set_ylabel("zeta   (0 = bed, 1 = surface)")
    ax1.set_title("vertical velocity profile")
    ax1.legend()
    fig.tight_layout()

## Try it yourself

Edit and re-run any cell — this is a live kernel:

- Change `nu` and `lambda_s` and watch the velocity profile change: more slip
  (`lambda_s`) tilts it, more viscosity (`nu`) shapes its curvature.
- `level=1` (one moment, a straight profile) vs `level=3` (more curvature).
- `ReconstructionSpec(order=2)` for a sharper front.
- `BC.Wall` → `BC.Extrapolation` on both tags, and watch mass leave the domain.

Only the pure-Python NumPy backend is available in the browser. For JAX, AMReX
or OpenFOAM, install locally or use a container — see the
[documentation](https://zoomylab.github.io/Zoomy/installation.html).